# Dimensionamento de Equipe de Capacidade Operacional

Este notebook responde à pergunta central de gestão:
**Quantos analistas são necessários para atender a demanda em cada cenário?**

Perguntas respondidas:
- Qual a capacidade resolutiva atual por analista?
- Quantos analistas são necessários para zerar o backlog em 30, 60 ou 90 dias?
- Como esse número muda nos 4 cenários de demanda?
- Qual o custo-benefício de expandir a equipe versus o custo do backlog crescente?

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_processing import tratar_dados
from src.metrics import (
    volume_diario_recebidos,
    volume_diario_resolvidos,
    capacidade_backlog_diaria,
    performance_media_por_analista
)

df = tratar_dados()

## 1. Capacidade Atual da Equipe

In [ ]:
vol_rec = volume_diario_recebidos(df)
vol_res = volume_diario_resolvidos(df)
backlog = capacidade_backlog_diaria(df)
perf_analista = performance_media_por_analista(df)

media_recebidos_dia = vol_rec['recebidos'].mean()
media_resolvidos_dia = vol_res['resolvidos'].mean()
backlog_acumulado_atual = backlog['saldo'].sum()

# Número de analistas ativos (com tickets fechados)
n_analistas_atuais = len(perf_analista)

# Capacidade resolutiva por analista por dia
capacidade_por_analista_dia = media_resolvidos_dia / n_analistas_atuais

print("=" * 55)
print("       CAPACIDADE ATUAL DA EQUIPE")
print("=" * 55)
print(f"  Analistas ativos           : {n_analistas_atuais}")
print(f"  Resolvidos/dia (total)     : {media_resolvidos_dia:.2f} tickets")
print(f"  Resolvidos/dia (por analista): {capacidade_por_analista_dia:.2f} tickets")
print(f"  Recebidos/dia              : {media_recebidos_dia:.2f} tickets")
print(f"  Backlog acumulado atual    : {backlog_acumulado_atual:.0f} tickets")
print("=" * 55)

## 2. Capacidade Resolutiva por Analista

Distribuição da produtividade individual (tickets/dia), estimada a partir do período histórico.

In [ ]:
# Tickets resolvidos por analista no período
df_fechados = df[df['status_ticket'] == 'Closed'].copy()
n_dias_historico = (df_fechados['data_fechamento'].max() - df_fechados['data_fechamento'].min()).days

tickets_por_analista = df_fechados.groupby('analista_responsavel').size().reset_index(name='tickets_fechados')
tickets_por_analista['tickets_por_dia'] = (tickets_por_analista['tickets_fechados'] / n_dias_historico).round(3)
tickets_por_analista = tickets_por_analista.sort_values('tickets_por_dia', ascending=False)

print(f"Período histórico: {n_dias_historico} dias")
print()
print(tickets_por_analista.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(tickets_por_analista['analista_responsavel'],
               tickets_por_analista['tickets_por_dia'],
               color='#4472C4', edgecolor='white', linewidth=1)
ax.axvline(capacidade_por_analista_dia, color='#d62728', linestyle='--',
           linewidth=1.5, label=f'Média: {capacidade_por_analista_dia:.3f} tickets/dia')
for bar, val in zip(bars, tickets_por_analista['tickets_por_dia']):
    ax.text(val + 0.001, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=9)
ax.set_title('Produtividade por Analista (tickets/dia)', fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Tickets fechados por dia')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Cálculo de Analistas Necessários por Cenário e Meta

Para cada combinação de cenário de demanda × prazo para zerar o backlog, calculamos o número mínimo de analistas.

**Fórmula:**
```
analistas_necessários = ⌈(demanda_diária + backlog / prazo_dias) / capacidade_por_analista⌉
```

In [ ]:
cenarios_demanda = {
    'A: -20%': media_recebidos_dia * 0.80,
    'B: Baseline': media_recebidos_dia * 1.00,
    'C: +20%': media_recebidos_dia * 1.20,
    'D: +50%': media_recebidos_dia * 1.50,
}

metas_dias = [30, 60, 90]

tabela = {}
for nome_cen, demanda in cenarios_demanda.items():
    linha = {}
    for prazo in metas_dias:
        # Capacidade necessária por dia:
        # = atender demanda diária + liquidar o backlog no prazo
        capacidade_necessaria = demanda + max(backlog_acumulado_atual, 0) / prazo
        n_analistas = math.ceil(capacidade_necessaria / capacidade_por_analista_dia)
        linha[f'Meta D+{prazo}'] = n_analistas
    tabela[nome_cen] = linha

df_tabela = pd.DataFrame(tabela).T
print("Analistas necessários por cenário × prazo para zerar backlog:")
print()
print(df_tabela.to_string())
print()
print(f"  (Equipe atual: {n_analistas_atuais} analistas)")

## 4. Heatmap de Analistas Necessários

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(
    df_tabela,
    annot=True, fmt='d', cmap='YlOrRd',
    linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Analistas necessários'}
)
ax.set_title(f'Analistas Necessários por Cenário × Prazo\n(Equipe atual: {n_analistas_atuais} analistas)',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Meta para zerar backlog')
ax.set_ylabel('Cenário de demanda')
plt.tight_layout()
plt.show()

## 5. Comparação: Equipe Atual vs Necessária por Cenário

Quantos analistas faltam (ou sobram) em cada cenário com meta de 60 dias.

In [ ]:
meta_referencia = 'Meta D+60'

fig, ax = plt.subplots(figsize=(9, 5))
nomes_cenarios = list(df_tabela.index)
necessarios = df_tabela[meta_referencia].values
x = np.arange(len(nomes_cenarios))
width = 0.35

b1 = ax.bar(x - width/2, [n_analistas_atuais] * len(nomes_cenarios), width,
            label='Equipe atual', color='#4472C4', alpha=0.8)
b2 = ax.bar(x + width/2, necessarios, width,
            label=f'Necessário ({meta_referencia})',
            color=['#2ca02c' if n <= n_analistas_atuais else '#d62728' for n in necessarios],
            alpha=0.85)

for bar, val in zip(b1, [n_analistas_atuais] * len(nomes_cenarios)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, str(val),
            ha='center', va='bottom', fontsize=10, fontweight='bold')
for bar, val in zip(b2, necessarios):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, str(val),
            ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(nomes_cenarios)
ax.set_ylabel('Número de analistas')
ax.set_title(f'Equipe Atual vs Necessária por Cenário (meta: {meta_referencia})',
             fontsize=13, fontweight='bold', pad=15)
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Análise de Custo-Benefício Simples

Comparamos o custo de contratar analistas adicionais versus o custo implícito do backlog não resolvido.

> Ajuste os valores de custo abaixo conforme a realidade da organização.

In [ ]:
CUSTO_ANALISTA_MES = 5_000
CUSTO_TICKET_BACKLOG = 50

linhas_custo = []
for nome_cen, demanda in cenarios_demanda.items():
    analistas_necessarios_60 = df_tabela.loc[nome_cen, 'Meta D+60']
    analistas_extras = max(0, analistas_necessarios_60 - n_analistas_atuais)

    # Backlog projetado em 60 dias SEM expansão
    saldo_diario_sem_expansao = demanda - media_resolvidos_dia
    backlog_60_sem_expansao = backlog_acumulado_atual + saldo_diario_sem_expansao * 60

    custo_expansao_2m = analistas_extras * CUSTO_ANALISTA_MES * 2  # 2 meses
    custo_backlog_2m = max(0, backlog_60_sem_expansao) * CUSTO_TICKET_BACKLOG

    linhas_custo.append({
        'Cenário': nome_cen,
        'Analistas extras': analistas_extras,
        'Custo expansão (2m)': f"R$ {custo_expansao_2m:,.0f}",
        'Backlog D+60 sem expansão': int(max(0, backlog_60_sem_expansao)),
        'Custo backlog (2m)': f"R$ {custo_backlog_2m:,.0f}",
        'Decisão': '✔ Expandir' if custo_expansao_2m < custo_backlog_2m else '○ Monitorar'
    })

df_custo = pd.DataFrame(linhas_custo).set_index('Cenário')
print(f"Premissas: analista = R$ {CUSTO_ANALISTA_MES:,}/mês | ticket em backlog = R$ {CUSTO_TICKET_BACKLOG}/mês")
print()
print(df_custo.to_string())

## 7. Tabela de Decisão de Recomendação Final

In [ ]:
print("=" * 70)
print("          TABELA DE DECISÃO — DIMENSIONAMENTO DE EQUIPE")
print("=" * 70)
print(f"  Equipe atual: {n_analistas_atuais} analistas")
print(f"  Capacidade atual: {media_resolvidos_dia:.1f} tickets/dia")
print(f"  Demanda atual:    {media_recebidos_dia:.1f} tickets/dia")
print()

for nome_cen, demanda in cenarios_demanda.items():
    necessario_30 = df_tabela.loc[nome_cen, 'Meta D+30']
    necessario_60 = df_tabela.loc[nome_cen, 'Meta D+60']
    necessario_90 = df_tabela.loc[nome_cen, 'Meta D+90']
    extras_60 = max(0, necessario_60 - n_analistas_atuais)

    print(f"  {nome_cen}")
    print(f"    Demanda: {demanda:.1f} tickets/dia")
    print(f"    Analistas para zerar em 30d: {necessario_30} | 60d: {necessario_60} | 90d: {necessario_90}")
    if extras_60 > 0:
        print(f"    → Requer +{extras_60} analista(s) para meta de 60 dias")
    else:
        print(f"    → Equipe atual suficiente para meta de 60 dias")
    print()

print("=" * 70)